In [1]:
!pip install -U wandb
!pip install -q transformers datasets evaluate scikit-learn librosa

import os
import torch
import numpy as np
import pandas as pd
import librosa
import evaluate
import wandb
from torch.utils.data import Dataset
from transformers import ASTForAudioClassification, ASTFeatureExtractor, Trainer, TrainingArguments
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("key")
wandb.login(key=wandb_key)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 70.4 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.25.0
    Uninstalling wandb-0.25.0:
      Successfully uninstalled wandb-0.25.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f2007692 (24f2007692-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
csv_path = '/kaggle/input/notebooks/kartikbajaj1322/data-generation-5-chunks-for-ast/ast_metadata.csv'
base_dir = '/kaggle/input/notebooks/kartikbajaj1322/data-generation-5-chunks-for-ast/ast_wav_data/'

if not os.path.exists(csv_path):
    csv_path = 'ast_metadata.csv'
    base_dir = './ast_wav_data'

df = pd.read_csv(csv_path)

labels = sorted(df['genre'].unique())
num_labels = len(labels)
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}

df['label'] = df['genre'].map(label2id)

feature_extractor = ASTFeatureExtractor.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')

class AudioDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(base_dir, row['filename'])
        
        y, _ = librosa.load(path, sr=16000)
        
        target_len = 16000 * 10
        if len(y) > target_len:
            y = y[:target_len]
        else:
            y = np.pad(y, (0, target_len - len(y)))
            
        features = feature_extractor(
            y,
            sampling_rate=16000,
            padding='max_length',
            max_length=1024,
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_values': features['input_values'][0],
            'label': int(row['label'])
        }


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [3]:
val_fold = 0

train_df = df[df['fold'] != val_fold]
val_df = df[df['fold'] == val_fold]

train_dataset = AudioDataset(train_df)
val_dataset = AudioDataset(val_df)
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")


Train samples: 8000, Val samples: 2000


In [4]:
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids, average="macro")


In [5]:
wandb.init(project='24f2007692-t12026', name='AST-Fold-Tune_2')

model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

args = TrainingArguments(
    output_dir="ast_temp",
    eval_strategy="epoch",
    save_strategy="epoch",
    
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to='wandb'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
val_res = trainer.evaluate()
print("Validation Results:", val_res)
wandb.finish()


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260329_082804-c3fxh3xr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run AST-Fold-Tune_2
wandb: ⭐️ View project at https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026
wandb: 🚀 View run at https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026/runs/c3fxh3xr


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1
1,No log,0.271023,0.906933
2,0.494433,0.191632,0.941111
3,0.494433,0.226084,0.936351
4,0.030376,0.175547,0.955892
5,0.030376,0.129294,0.969388
6,0.001898,0.123737,0.968330
7,0.001898,0.125140,0.967838
8,0.000172,0.125349,0.967838


... (7 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: updating run metadata


Validation Results: {'eval_loss': 0.12929415702819824, 'eval_f1': 0.969387952050781, 'eval_runtime': 100.2921, 'eval_samples_per_second': 19.942, 'eval_steps_per_second': 1.246, 'epoch': 8.0}


wandb: uploading history steps 13-13, summary, console lines 10-10
wandb: 
wandb: Run history:
wandb:                 eval/f1 ▁▅▄▆█████
wandb:               eval/loss █▄▆▃▁▁▁▁▁
wandb:            eval/runtime █▃▃▄▅▁▆▁▂
wandb: eval/samples_per_second ▁▆▆▅▄█▃█▇
wandb:   eval/steps_per_second ▁▆▆▅▄█▃█▆
wandb:             train/epoch ▁▂▂▃▄▄▅▆▆▇████
wandb:       train/global_step ▁▂▂▃▄▄▅▆▆▇████
wandb:         train/grad_norm ▂█▁▁
wandb:     train/learning_rate █▅▂▁
wandb:              train/loss █▁▁▁
wandb: 
wandb: Run summary:
wandb:                 eval/f1 0.96939
wandb:               eval/loss 0.12929
wandb:            eval/runtime 100.2921
wandb: eval/samples_per_second 19.942
wandb:   eval/steps_per_second 1.246
wandb:              total_flos 4.33841139351552e+18
wandb:             train/epoch 8
wandb:       train/global_step 2000
wandb:         train/grad_norm 0.00722
wandb:     train/learning_rate 0.0
wandb:                      +5 ...
wandb: 
wandb: 🚀 View run AST-Fold-Tune_2 at: htt

In [6]:
wandb.init(project='24f2007692-t12026', name='AST-Full-Data_2')

full_dataset = AudioDataset(df)

final_model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

final_args = TrainingArguments(
    output_dir="ast_model",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to='wandb'
)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    train_dataset=full_dataset
)

final_trainer.train()

final_trainer.save_model('ast_model_final')
wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260329_113311-oe5qe16l
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run AST-Full-Data_2
wandb: ⭐️ View project at https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026
wandb: 🚀 View run at https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026/runs/oe5qe16l


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Step,Training Loss
500,0.261189
1000,0.027195


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: updating run metadata
wandb: uploading history steps 2-2, summary
wandb: 
wandb: Run history:
wandb:         train/epoch ▁▆█
wandb:   train/global_step ▁▆█
wandb:     train/grad_norm █▁
wandb: train/learning_rate █▁
wandb:          train/loss █▁
wandb: 
wandb: Run summary:
wandb:               total_flos 2.7115071209472e+18
wandb:              train/epoch 4
wandb:        train/global_step 1252
wandb:          train/grad_norm 0.15319
wandb:      train/learning_rate 0.0
wandb:               train/loss 0.02719
wandb:               train_loss 0.11554
wandb:            train_runtime 6360.4447
wandb: train_samples_per_second 6.289
wandb:   train_steps_per_second 0.197
wandb: 
wandb: 🚀 View run AST-Full-Data_2 at: https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026/runs/oe5qe16l
wandb: ⭐️ View project at: https://wandb.ai/24f2007692-indian-institute-of-technology-madras/24f2007692-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) an